# 最小联网能力测试（模型 + 外网）

这个 Notebook 用最小步骤判断两件事：
1. 当前环境是否具备基础外网访问能力。
2. 当前模型是否真正触发了 `web_search_preview` 联网工具调用。

判定：
- `PASS`：检测到 `web_search` 工具调用事件。
- `FAIL`：未检测到工具调用，或请求失败。

In [20]:
import os
import json
import time
import requests

# 可按需改成 https://www.gstatic.com/generate_204
TEST_URL = "https://httpbin.org/get"
TIMEOUT_SEC = 5

# 模型配置
API_KEY = "sk-5ynRCf5QHB8Z8xDYP20wmXZSX2wQoGA7LykQ3PzMvQik0F88"
BASE_URL = os.getenv("LLM_BASE_URL", "https://api.chatanywhere.tech/v1")
MODEL = os.getenv("LLM_MODEL", "gpt-5-nano")

print("BASE_URL:", BASE_URL)
print("MODEL:", MODEL)
print("API_KEY loaded:", bool(API_KEY))

BASE_URL: https://api.chatanywhere.tech/v1
MODEL: gpt-5-nano
API_KEY loaded: True


In [22]:
def probe_internet(url: str, timeout: int = 5):
    start = time.perf_counter()
    try:
        resp = requests.get(url, timeout=timeout)
        latency_ms = round((time.perf_counter() - start) * 1000, 2)
        ok = 200 <= resp.status_code < 400
        return {
            "online": ok,
            "status_code": resp.status_code,
            "latency_ms": latency_ms,
            "error": None,
        }
    except Exception as e:
        latency_ms = round((time.perf_counter() - start) * 1000, 2)
        return {
            "online": False,
            "status_code": None,
            "latency_ms": latency_ms,
            "error": f"{type(e).__name__}: {e}",
        }

network_result = probe_internet(TEST_URL, timeout=TIMEOUT_SEC)
print(json.dumps(network_result, ensure_ascii=False, indent=2))

# 最小可断言网络检查
assert network_result["online"] is True, f"外网不可达: {network_result['error']}"

{
  "online": true,
  "status_code": 200,
  "latency_ms": 1309.26,
  "error": null
}


In [ ]:
pip install --upgrade openai

In [23]:
from openai import OpenAI

if not API_KEY:
    raise ValueError("请先设置环境变量 LLM_API_KEY 或 OPENAI_API_KEY")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

resp = client.responses.create(
    model=MODEL,
    tools=[{"type": "web_search_preview"}],
    input="请联网搜索今天全球最受关注的3条科技新闻，并给出每条来源域名。",
)

raw = resp.model_dump() if hasattr(resp, "model_dump") else dict(resp)
output_text = raw.get("output_text", "")
items = raw.get("output", []) if isinstance(raw.get("output"), list) else []
web_events = [x for x in items if isinstance(x, dict) and "web_search" in str(x.get("type", ""))]

print("response.id:", raw.get("id"))
print("response.model:", raw.get("model"))
print("response.created_at:", raw.get("created_at"))
print("web_search_events:", len(web_events))
print("\n--- output_text (前800字符) ---\n")
print((output_text or "")[:800])

if len(web_events) > 0:
    print("\nPASS: 检测到真实联网工具调用")
else:
    print("\nFAIL: 未检测到联网工具调用（可能回退或网关不支持）")

# 最小可断言模型联网检查
assert len(web_events) > 0, "未检测到 web_search 工具调用事件"

response.id: resp_010149ebf2100d9e00699dc872bde081968c46df51930e9110
response.model: gpt-5-nano-2025-08-07
response.created_at: 1771948146.0
web_search_events: 7

--- output_text (前800字符) ---



PASS: 检测到真实联网工具调用


## 判定标准与常见失败原因

- PASS：出现 `web_search_events > 0` 且断言通过，说明模型端真实触发了联网工具。
- FAIL：
  - 模型/网关不支持 `responses` 或 `web_search_preview`。
  - API Key 权限不足。
  - 外网受限（可先看上一个单元 `online` 是否为 `true`）。

> 这是最小测试案例：先测网络，再测模型工具调用，且都带断言。